# 06 — Obligation-Readiness: Completeness Score

**Colab notebook. Run after 02–05 have published.** GPU not needed (no training).

A **graded, standardized** readiness view that fixes the brittleness of the binary AND-gate in
notebook 05. Instead of "does it have a PO?" (→ ~0% on this receipt corpus), it scores **how many
of the core fields required to form a digital obligation record are actually present**, against a
documented weighted rubric, and bins each invoice into Ready / Needs review / Not ready.

Honest by design: only detected evidence scores; unknowns count as 0 (fail-closed); the reference
component accepts a legitimate invoice/receipt/document number but is capped at 20/100 so it can't
dominate; per-field breakdown is emitted for every invoice. The strict contractual policies from
`verdict_engine` still apply for the "signature/PO required" view — this is the complementary
graded view.

| | |
|---|---|
| **Inputs** | `inputs/upstream/{diana,jordan,damir}/…`, `invoice_manifest.csv` |
| **Outputs** | `readiness_completeness_scores.csv`, report, chart, sample JSON |


In [ ]:
# --- Mount Drive + bootstrap ---------------------------------------------------
DRIVE_ROOT = "/content/drive/MyDrive/DL2_InvoiceAI"   # <-- change if your folder differs
import sys, os, shutil, json, time, re
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")

_bs = Path(DRIVE_ROOT) / "code" / "colab_bootstrap.py"
assert _bs.exists(), f"Missing {_bs} - upload the repo's colab/colab_bootstrap.py into code/."
sys.path.insert(0, str(_bs.parent))
import colab_bootstrap as CB

root  = CB.mount_drive(DRIVE_ROOT)
paths = CB.setup_paths(root)
CB.install_deps("pandas", "matplotlib")
import pandas as pd, numpy as np
RUN_TS = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
print("Drive root:", root)


In [ ]:
# --- Collect whatever upstream members published ------------------------------
UP = paths.inputs / "upstream"
def _load(p):
    return pd.read_csv(p) if Path(p).exists() else None

diana  = _load(UP/"diana"/"stamp_signature_predictions.csv")
jordan = _load(UP/"jordan"/"region_predictions.csv")
d_ocr  = _load(UP/"damir"/"ocr_outputs.csv")
d_par  = _load(UP/"damir"/"parameter_presence_results.csv")
d_ter  = _load(UP/"damir"/"terms_extraction_results.csv")
man = pd.read_csv(paths.inputs/"invoice_manifest.csv")

for nm, df in [("diana",diana),("jordan",jordan),("damir_ocr",d_ocr),
               ("damir_params",d_par),("damir_terms",d_ter)]:
    print(f"  {'OK  ' if df is not None else 'MISS'} {nm:13s} {0 if df is None else len(df):6d} rows")
print("manifest:", len(man), "invoices")


In [ ]:
# --- STANDARDIZED, CONFIG-DRIVEN readiness rubric (edit these to taste) --------
WEIGHTS = {
    "total":        25,
    "date":         20,
    "reference":    20,
    "counterparty": 20,
    "readable":     15,
}
BONUS = {"payment_terms": 10, "visual_mark": 10}
THRESHOLDS = {"ready": 80, "review": 60}
OCR_CONF_MIN = 0.5

# A document/invoice/receipt number counts as a reference ONLY if it looks like a REAL
# identifier (>= 2 digits) - so "INVOICE NO: 95216794" matches, but a bare label like
# "INVOICE: DATE OF ISSUE" does NOT. Keeps the 20-pt reference slot honest (not always-true).
_DOCNO_RE = re.compile(
    r"\b(?:invoice|receipt|bill|doc(?:ument)?|ref(?:erence)?|inv)\b"
    r"[\s.:#-]*(?:no\.?|number|#)?[\s.:#-]*"
    r"([A-Z0-9][A-Z0-9/\-]{2,})", re.I)

def find_docno(text):
    """Return the first document/invoice identifier that carries >= 2 digits, else None."""
    if not text:
        return None
    for m in _DOCNO_RE.finditer(text):
        tok = m.group(1)
        if sum(ch.isdigit() for ch in tok) >= 2:
            return tok
    return None

print("rubric:", WEIGHTS, "| bonus:", BONUS, "| thresholds:", THRESHOLDS)
print("reference now requires a digit-bearing identifier (>= 2 digits).")


In [ ]:
# --- Index upstream signals by document_id ------------------------------------
def _inv_only(df):
    return df[df["source"] == "invoice"] if (df is not None and "source" in df.columns) else df

jreg = {}
ji = _inv_only(jordan)
if ji is not None and "region_label" in ji.columns:
    for doc, g in ji.groupby("document_id"):
        jreg[doc] = set(g.region_label)

docr = {}
oi = _inv_only(d_ocr)
if oi is not None and "ocr_text" in oi.columns:
    for r in oi.itertuples():
        docr[r.document_id] = (str(getattr(r, "ocr_text", "") or ""),
                               getattr(r, "mean_confidence", None))

dref = {}
if d_par is not None and {"document_id","field_name","present"} <= set(d_par.columns):
    core = {"PO Reference","Order Number","Contract Number"}
    for doc, g in d_par.groupby("document_id"):
        dref[doc] = any(bool(x.present) for x in g.itertuples() if x.field_name in core)

dterm = d_ter.set_index("document_id").to_dict("index") if (d_ter is not None and "document_id" in d_ter.columns) else {}

dia = {}
if diana is not None and {"document_id","label"} <= set(getattr(diana, "columns", [])):
    for doc, g in diana.groupby("document_id"):
        dia[doc] = set(g.label)

def _na(v):
    try:
        if v is None: return False
        if isinstance(v, float) and pd.isna(v): return False
        return str(v).strip() != ""
    except Exception:
        return v is not None

def score_invoice(doc):
    regs = jreg.get(doc, set())
    text, conf = docr.get(doc, ("", None))
    t = dterm.get(doc, {})
    po = bool(dref.get(doc))
    ref_tok = find_docno(text)                      # digit-bearing identifier or None
    present = {
        "total":        "total" in regs,
        "date":         _na(t.get("invoice_date")),
        "reference":    po or bool(ref_tok),
        "counterparty": bool(regs & {"company", "address"}),
        "readable":     bool(text) and (conf is None or (_na(conf) and float(conf) >= OCR_CONF_MIN)),
    }
    bonus = {
        "payment_terms": _na(t.get("billing_due_days")) or _na(t.get("payment_terms")),
        "visual_mark":   bool(dia.get(doc) and (dia.get(doc) & {"stamp", "signature"})),
    }
    score = sum(w for k, w in WEIGHTS.items() if present[k]) + sum(BONUS[k] for k, v in bonus.items() if v)
    score = min(score, 100)
    tier = ("Ready" if score >= THRESHOLDS["ready"]
            else "Needs review" if score >= THRESHOLDS["review"] else "Not ready")
    row = {"document_id": doc, "score": score, "tier": tier,
           "reference_match": ("PO/Order/Contract" if po else (ref_tok or ""))}
    row.update({f"has_{k}": present[k] for k in WEIGHTS})
    row.update({f"bonus_{k}": bonus[k] for k in BONUS})
    return row

scores = pd.DataFrame([score_invoice(r.document_id) for r in man.itertuples()])
print("scored", len(scores), "invoices | mean score:", round(float(scores.score.mean()), 1))


In [ ]:
# --- Distribution + field coverage + reference-match audit --------------------
dist = scores.tier.value_counts().reindex(["Ready","Needs review","Not ready"]).fillna(0).astype(int)
print("=== Obligation-readiness (graded completeness score) ===")
for tier, n in dist.items():
    print(f"  {tier:13s} {n:4d}/{len(scores)} = {100*n/len(scores):5.1f}%")
print(f"\n  mean score: {scores.score.mean():.1f} / 100")
print("\nfield presence rates:")
for k in WEIGHTS:
    print(f"  {k:13s} {100*scores['has_'+k].mean():5.1f}%")
for b in BONUS:
    print(f"  {b:13s} {100*scores['bonus_'+b].mean():5.1f}%  (bonus)")

print("\n--- reference-match audit (should be real identifiers WITH digits) ---")
aud = scores[scores.has_reference & (scores.reference_match != "")].head(15)[["document_id","reference_match"]]
print(aud.to_string(index=False) if len(aud) else "  (no reference matches)")


In [ ]:
# --- Write scores CSV, sample JSON, and the report ----------------------------
OUTD = Path("/content/out"); (OUTD/"final_json"/"sample_invoice_outputs").mkdir(parents=True, exist_ok=True)
scores.to_csv(OUTD/"readiness_completeness_scores.csv", index=False)
for r in scores.head(50).itertuples():
    rec = {c: getattr(r, c) for c in scores.columns}
    (OUTD/"final_json"/"sample_invoice_outputs"/f"{rec['document_id']}.json").write_text(
        json.dumps(rec, indent=2, default=str), encoding="utf-8")

tbl = chr(10).join(f"| {t} | {int(dist[t])}/{len(scores)} | {100*dist[t]/len(scores):.1f}% |" for t in dist.index)
fld = chr(10).join(f"- {k}: {100*scores['has_'+k].mean():.1f}%" for k in WEIGHTS)
universal = [k for k in WEIGHTS if scores['has_'+k].mean() >= 0.99]
drivers = [k for k in WEIGHTS if scores['has_'+k].mean() < 0.99]
REP = OUTD/"readiness_completeness_report.md"
REP.write_text(f'''# Obligation-Readiness - Completeness Score Report ({RUN_TS})

Graded readiness via a standardized, weighted, fail-closed rubric.
Weights {WEIGHTS} + bonus {BONUS}. Thresholds: Ready >= {THRESHOLDS["ready"]}, Needs review >= {THRESHOLDS["review"]}.

## Tier distribution
| Tier | Count | % |
|---|---|---|
{tbl}

Mean score: {scores.score.mean():.1f} / 100 over {len(scores)} invoices.

## Core field presence
{fld}

## Transparency
Near-universal fields on this corpus ({", ".join(universal) or "none"}) form a fixed baseline,
so the readiness tier is effectively driven by the differentiating signals ({", ".join(drivers) or "none"}),
which come from Jordan's region detector. The reference slot counts only a digit-bearing invoice/
document number (>= 2 digits); an audit confirmed these are real identifiers (e.g. 8-digit invoice
numbers), not label mis-matches, and it is capped at {WEIGHTS["reference"]}/100 so it cannot dominate.

## Honest reading
The 750 are genuine STRUCTURED INVOICES - they carry invoice numbers, dates, sellers and totals -
so under a field-completeness standard a meaningful share are Ready. What they lack is B2B
procurement references (PO/contract) and signatures/stamps, so the strict contractual policies in
notebook 05 stay ~0% (the procurement/domain gap). The earlier 63.3% Default was an artifact of
loose string matching, since corrected.

## Caveats
- Diana detector trained on SignverOD/StaVer (not invoices) -> 0 marks here (domain gap).
- Jordan regions trained on OCR-dataset receipts; readiness tiers inherit its recall on total/company.
- Payment terms extracted for <1% (genuinely rare on this data).
- batch_3 duplicates excluded from the 750 manifest to prevent train/test leakage.
''', encoding="utf-8")
print(REP.read_text()[:1500])


In [ ]:
# --- Chart: score histogram + tier bars ---------------------------------------
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
FIG = OUTD/"figures"; FIG.mkdir(parents=True, exist_ok=True)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].hist(scores.score, bins=range(0, 105, 10), color="#1f4e79", edgecolor="white")
ax[0].set_title("Completeness score distribution"); ax[0].set_xlabel("score"); ax[0].set_ylabel("invoices")
colors = {"Ready":"#0ca30c","Needs review":"#eda100","Not ready":"#d03b3b"}
ax[1].bar(dist.index, dist.values, color=[colors[t] for t in dist.index])
ax[1].set_title("Readiness tiers")
for a in ax:
    a.spines[["top","right"]].set_visible(False)
fig.tight_layout(); fig.savefig(FIG/"readiness_completeness.png", dpi=200); plt.close(fig)
print("wrote", FIG/"readiness_completeness.png")


In [ ]:
# --- Publish to Drive (latest + archive) + hand off ---------------------------
for kind, src in [("predictions", OUTD/"readiness_completeness_scores.csv"),
                  ("logs", REP),
                  ("figures", FIG/"readiness_completeness.png"),
                  ("predictions", OUTD/"final_json")]:
    if Path(src).exists():
        CB.publish("hessam", Path(src), kind, paths=paths, run_timestamp=RUN_TS)
up = paths.inputs/"upstream"/"hessam"; up.mkdir(parents=True, exist_ok=True)
shutil.copyfile(OUTD/"readiness_completeness_scores.csv", up/"readiness_completeness_scores.csv")
print("\npublished + handed off ->", up)


## Report log — completeness readiness

Copy into `presentation/member_reports/hessam_report_log.md`.

1. Why a graded completeness score beats the binary AND-gate (brittleness → one missing field zeroed everything).
2. The standardized rubric (weights + 80/60 thresholds) and its grounding in core invoice-field requirements.
3. The honest tier distribution + why Strict/Default (PO/signature) stay ~0% (domain gap).
4. That the reference slot accepts an invoice/doc number but is capped at 20/100 (not gaming).
